# 03 — OCR de matrícules (mètode UHU, multi-país)

Reproducció en Python del pipeline d'OCR del repositori
[UHU-VC_SegmentacionReconocimientoMatriculas](https://github.com/byLiTTo/byLiTTo/UHU-VC_SegmentacionReconocimientoMatriculas/),
adaptat als nostres retalls de `data/processed/` i estès per cobrir matrícules de **diversos països europeus** (ES, FR, IT, PL, CZ, RS, DE...).

## Esquema del mètode UHU

1. **Canal vermell** + binarització amb **Otsu** (`Ib = R < umbral`).
2. **Eliminació de regions sorolloses** (doble passada de `bwareaopen`).
3. **Etiquetatge** + filtre per **línies de terços** (blobs que travessen H/3 i 2H/3).
4. **Top (N+1) per àrea + descart de l'esquerre** (logotip UE).
5. **Reconeixement per template matching** amb correlació normalitzada.

## Adaptacions per a Europa

- **Alfabet complet 0-9 + A-Z** (36 classes) per cobrir vocals (no només l'alfabet ES restringit).
- **Multi-font**: Arial Black com a fallback, però accepta FE-Schrift si la poses a `fonts/FE-Schrift.ttf` (font oficial alemanya, molt propera a la majoria EU).
- **Iteració sobre `NUM_OBJETOS`** (provem 5/6/7/8 caràcters) — diferents països tenen llargades diferents.
- **Validador multi-país**: patrons regex per a ES, FR/IT, PL, CZ, RS, DE + fallback genèric (5-8 chars amb mix de lletres i dígits).

## Rebuig

Un retall es **rebutja** si:
- La segmentació no produeix un nombre vàlid de caràcters per cap dels `NUM_OBJETOS_TRY`, o
- La confiança mitjana del reconeixement és per sota de `MIN_CONFIDENCE`, o
- La cadena reconeguda no compleix cap patró de matrícula ni la regla estructural (mix lletres/dígits, sense repeticions excessives).

In [ ]:
import re
import random
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

random.seed(0)
np.random.seed(0)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['image.cmap'] = 'gray'

CROPS_DIR = Path('data/processed')
crops = sorted(CROPS_DIR.glob('*.png'))
print(f'OpenCV {cv2.__version__} — {len(crops)} retalls a {CROPS_DIR}')

## 1. Configuració

`NUM_OBJETOS` és el nombre esperat de caràcters de la matrícula (7 per al format espanyol modern: 4 dígits + 3 lletres).

`ALFABET` és el conjunt de caràcters permesos a matrícules espanyoles (no inclou vocals, Ñ, Q, etc., per evitar confusions).

In [ ]:
# Provem aquests valors de NumObjetos en ordre (el primer que doni una placa vàlida guanya)
NUM_OBJETOS_TRY = [7, 6, 8, 5]
NUM_OBJETOS = NUM_OBJETOS_TRY[0]

# Alfabet complet 0-9 + A-Z (36 classes) per cobrir matrícules amb vocals (PL, CZ, RS, DE...)
ALFABET = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Rotacions per plantilla
ANGLES_DEG = np.linspace(-9, 9, 7)

# Llindar de confiança mitjana per acceptar una predicció
MIN_CONFIDENCE = 0.35

# Patrons per país: una matrícula s'accepta si coincideix amb algun
# (de més específic a més genèric)
PLATE_PATTERNS = {
    'es_modern':  re.compile(r'^\d{4}[A-Z]{3}$'),                # ES: 1234ABC
    'fr_it':      re.compile(r'^[A-Z]{2}\d{3}[A-Z]{2}$'),        # FR/IT: AB123CD
    'rs':         re.compile(r'^[A-Z]{2}\d{3,4}[A-Z]{1,2}$'),    # RS: BG123AB
    'pl':         re.compile(r'^[A-Z]{2,3}\d{4,5}[A-Z]?$'),      # PL: WA12345
    'cz':         re.compile(r'^\d[A-Z]\d[A-Z]\d{4}$'),        # CZ modern
    'de_generic': re.compile(r'^[A-Z]{1,3}\d{1,4}[A-Z]{0,2}$'),  # DE compacte
    # Fallback: 5-8 chars alfanumèrics amb almenys una lletra I un dígit
    'generic_eu': re.compile(r'^(?=.*[A-Z])(?=.*\d)[A-Z0-9]{5,8}$'),
}

print(f'Alfabet ({len(ALFABET)} chars): {ALFABET}')
print(f'NumObjetos a provar: {NUM_OBJETOS_TRY}')
print(f'Patrons ({len(PLATE_PATTERNS)}): {list(PLATE_PATTERNS.keys())}')
print(f'Min confiança mitjana: {MIN_CONFIDENCE}')

## 2. Fase 1 — Binarització amb canal R + Otsu

A UHU usen el canal **vermell** perquè els caràcters negres sobre fons blanc generen molt contrast en aquest canal (text → R baix, fons → R alt). Apliquem Otsu sobre R i invertim per tenir els caràcters en blanc.

`OpenCV usa ordre BGR`, així que el canal R és `img[:, :, 2]`.

In [ ]:
def red_channel_otsu(img_bgr):
    R = img_bgr[:, :, 2]
    umbral, _ = cv2.threshold(R, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Ib = R < umbral  (caràcters fosc → True)
    Ib = (R < umbral).astype(np.uint8) * 255
    return Ib, umbral

## 3. Fase 2 — Eliminació de regions sorolloses (geomètrica)

UHU fa dues passades de `bwareaopen`: la primera descarta components puntiformes (< 0.1 % imatge), la segona descarta tot el que sigui més petit que `àrea_màxima/5`.

**Adaptació nostra**: els nostres crops inclouen marc del cotxe al voltant de la matrícula. Aquest marc sovint forma el component més gran, i la regla `max/5` acaba eliminant **tots** els caràcters (cada un és < 1/5 del marc). Substituïm el segon pas per un **filtre geomètric**: només mantenim components amb mida i forma compatibles amb un caràcter:

- Alçada entre 20 % i 90 % de l'alçada del crop.
- Amplada < 30 % de l'amplada del crop (descarta barres horitzontals).
- Aspect ratio (w/h) entre 0.10 i 1.5.
- Densitat (àrea/bbox) ≥ 15 %.

Així el marc gegant es descarta sense menjar-se els caràcters.

In [ ]:
def bwareaopen(bw, min_size):
    # OpenCV equivalent de bwareaopen de MATLAB.
    n, labels, stats, _ = cv2.connectedComponentsWithStats(bw, connectivity=8)
    out = np.zeros_like(bw)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= min_size:
            out[labels == i] = 255
    return out


# Llindars geomètrics esperats per a caràcters dins d'un crop de matrícula:
# - alçada: entre 20 % i 90 % de l'alçada del crop
# - amplada: com a màxim 30 % de l'amplada del crop (els marcs grans queden fora)
# - aspect ratio (w/h): entre 0.10 i 1.5 (caràcters són alts i estrets)
# - densitat (àrea/bbox): mínim 15 % (descarta rectangles buits del marc)
CHAR_H_MIN_FRAC, CHAR_H_MAX_FRAC = 0.20, 0.90
CHAR_W_MAX_FRAC                  = 0.30
CHAR_AR_MIN, CHAR_AR_MAX         = 0.10, 1.5
CHAR_DENSITY_MIN                 = 0.15


def remove_noise(Ib):
    """Elimina soroll + filtre geomètric de caràcters.

    Substitueix el segon pas de UHU (`max_area/5`) per un filtre basat en
    proporcions: els nostres crops inclouen marc del cotxe i el component més
    gran sovint NO és un caràcter, així que `max_area/5` deixa zero caràcters.
    """
    N, M = Ib.shape

    # Pas 1 (UHU): treu components puntiformes (< 0.1 % de la imatge)
    Ib2 = bwareaopen(Ib, round(0.001 * N * M))
    if Ib2.sum() == 0:
        return Ib2

    # Pas 2 (modificat): filtre geomètric en lloc de max_area/5
    n, labels, stats, _ = cv2.connectedComponentsWithStats(Ib2, connectivity=8)
    out = np.zeros_like(Ib2)

    for i in range(1, n):
        x = stats[i, cv2.CC_STAT_LEFT]
        y = stats[i, cv2.CC_STAT_TOP]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]
        area = stats[i, cv2.CC_STAT_AREA]

        # Alçada: dins el rang típic de caràcter
        if not (CHAR_H_MIN_FRAC * N <= h <= CHAR_H_MAX_FRAC * N):
            continue
        # Amplada: descarta barres horitzontals (marc, banda inferior...)
        if w > CHAR_W_MAX_FRAC * M:
            continue
        # Aspect ratio
        ar = w / h if h > 0 else 0
        if not (CHAR_AR_MIN <= ar <= CHAR_AR_MAX):
            continue
        # Densitat dins el bounding box (descarta rectangles buits)
        density = area / (w * h) if w * h > 0 else 0
        if density < CHAR_DENSITY_MIN:
            continue

        out[labels == i] = 255

    return out

## 4. Fase 3 — Etiquetatge + filtre per línies de terços

Calculem etiquetes amb connectivitat-4 (com a UHU, més restrictiva i evita unir caràcters propers). Després ens quedem només amb els blobs que apareixen alhora a la fila `H/3` i a la fila `2H/3`: els caràcters de la matrícula travessen tota la franja central; soroll com vores, cargols o text superior/inferior no.

In [ ]:
def label_and_filter_thirds(bw):
    # Etiquetatge connectivitat-4 (com MATLAB bwlabel(I, 4))
    n, labels, stats, _ = cv2.connectedComponentsWithStats(bw, connectivity=4)
    H, W = bw.shape

    fila_sup = H // 3
    fila_inf = fila_sup * 2

    lbl_top = set(np.unique(labels[fila_sup, :]).tolist())
    lbl_bot = set(np.unique(labels[fila_inf, :]).tolist())
    common  = (lbl_top & lbl_bot) - {0}   # treu el fons

    Ipresentes = np.zeros_like(labels)
    for lab in common:
        Ipresentes[labels == lab] = lab
    return Ipresentes, stats, common

## 5. Fase 4 — Top (N+1) per àrea + descart de l'esquerre

Dels blobs que han passat el filtre de terços, ens quedem amb les `NumObjetos+1` regions de més àrea. Després re-etiquetem (la nova etiqueta 1 serà la més a l'esquerra per ordre de barrido de `bwlabel`) i la descartem: és el logotip UE blau.

In [ ]:
def keep_top_and_discard_leftmost(Ipresentes, stats, common, num_objetos):
    if len(common) == 0:
        return np.zeros_like(Ipresentes), []

    # Àrees dels blobs supervivents
    items = [(lab, stats[lab, cv2.CC_STAT_AREA]) for lab in common]
    items.sort(key=lambda t: -t[1])
    keep_n = min(num_objetos + 1, len(items))
    threshold_area = items[keep_n - 1][1]

    I_filtrada = np.zeros_like(Ipresentes, dtype=np.uint8)
    for lab, area in items:
        if area >= threshold_area:
            I_filtrada[Ipresentes == lab] = 255

    # Re-etiquetar i descartar la component més a l'esquerra (logotip UE)
    n, labels2, stats2, _ = cv2.connectedComponentsWithStats(I_filtrada, connectivity=4)
    if n <= 1:
        return labels2, []

    # Ordenar per coord. X (esquerre primer)
    comps = [(i, stats2[i, cv2.CC_STAT_LEFT]) for i in range(1, n)]
    comps.sort(key=lambda t: t[1])
    leftmost = comps[0][0]
    labels2[labels2 == leftmost] = 0

    # Llista de bounding boxes ordenats L→R
    final = [(stats2[i, cv2.CC_STAT_LEFT],
              stats2[i, cv2.CC_STAT_TOP],
              stats2[i, cv2.CC_STAT_WIDTH],
              stats2[i, cv2.CC_STAT_HEIGHT],
              i)
             for i in range(1, n) if i != leftmost]
    final.sort(key=lambda b: b[0])
    return labels2, final

## 6. Extracció de les ROI de cada caràcter

Per cada bounding box retallem la regió binària del caràcter.

In [ ]:
def extract_char_rois(labels, bboxes):
    rois = []
    for (x, y, w, h, lab) in bboxes:
        roi = (labels[y:y+h, x:x+w] == lab).astype(np.uint8) * 255
        rois.append(roi)
    return rois

def segment_plate(img_bgr, num_objetos=NUM_OBJETOS):
    # Fases 1-4 + extracció de ROIs. Retorna també les imatges intermèdies.
    Ib, umbral       = red_channel_otsu(img_bgr)
    Iclean           = remove_noise(Ib)
    Ipresentes, stats, common = label_and_filter_thirds(Iclean)
    Ietiq, bboxes    = keep_top_and_discard_leftmost(Ipresentes, stats, common, num_objetos)
    rois             = extract_char_rois(Ietiq, bboxes)
    return {
        'Ib':        Ib,
        'Iclean':    Iclean,
        'Ietiq':     (Ietiq > 0).astype(np.uint8) * 255,
        'labels':    Ietiq,
        'bboxes':    bboxes,
        'rois':      rois,
        'n_chars':   len(rois),
    }

## 7. Visualització de la segmentació sobre una mostra

In [ ]:
def show_segmentation_stages(img_bgr, res, title=''):
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(title, fontsize=11, fontweight='bold')
    axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(res['Ib']);     axes[1].set_title('R < Otsu');           axes[1].axis('off')
    axes[2].imshow(res['Iclean']); axes[2].set_title('Sense soroll');       axes[2].axis('off')
    axes[3].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    for (x, y, w, h, _) in res['bboxes']:
        axes[3].add_patch(plt.Rectangle((x, y), w, h, edgecolor='lime', facecolor='none', linewidth=2))
    axes[3].set_title(f"{res['n_chars']} caràcters"); axes[3].axis('off')
    plt.tight_layout(); plt.show()

sample_paths = random.sample(crops, 6)
for p in sample_paths:
    img = cv2.imread(str(p))
    res = segment_plate(img)
    show_segmentation_stages(img, res, title=p.name)

## 8. Generació de plantilles sintètiques (multi-font)

UHU usen `Plantillas.mat` (no fàcilment reutilitzable). Generem les nostres plantilles renderitzant cada caràcter amb les fonts disponibles del sistema.

**Fonts buscades en ordre**: `fonts/FE-Schrift.ttf` (font oficial alemanya, ideal per a la majoria de matrícules EU — la pots descarregar gratuïtament i posar-la al projecte), `fonts/EuroPlate.ttf`, `Arial Black` (fallback macOS), DejaVu Sans Bold (fallback Linux).

Si tens **múltiples fonts disponibles**, generem plantilles per cadascuna i mantenim totes les variants — el matching es queda amb la millor coincidència entre totes. Així cobrim millor les diferències tipogràfiques entre països.

Per cada plantilla generem **7 versions rotades** (−9° a +9°) per donar robustesa a inclinació residual.

In [ ]:
# Llista de fonts candidates a provar (en ordre de preferència).
# Posa fonts/FE-Schrift.ttf al projecte si la tens — és la font oficial alemanya
# i molt propera a la majoria de matrícules EU. Si no, fem servir Arial Black.
FONT_CANDIDATES = [
    Path('fonts/FE-Schrift.ttf'),
    Path('fonts/EuroPlate.ttf'),
    Path('assets/fonts/FE-Schrift.ttf'),
    Path('/System/Library/Fonts/Supplemental/Arial Black.ttf'),
    Path('/Library/Fonts/Arial Black.ttf'),
    Path('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'),
]

def find_available_fonts():
    found = [str(p) for p in FONT_CANDIDATES if p.exists()]
    if not found:
        raise FileNotFoundError(
            'Cap font trobada. Posa FE-Schrift.ttf a fonts/ o instal·la Arial Black.'
        )
    return found

TEMPLATE_H   = 60
TEMPLATE_PAD = 4

def render_char(ch, font_path, height=TEMPLATE_H, pad=TEMPLATE_PAD):
    font = ImageFont.truetype(font_path, height)
    bbox = font.getbbox(ch)
    w_txt = bbox[2] - bbox[0]
    h_txt = bbox[3] - bbox[1]
    canvas_w = max(1, w_txt + 2 * pad)
    canvas_h = max(1, h_txt + 2 * pad)
    img = Image.new('L', (canvas_w, canvas_h), 0)
    ImageDraw.Draw(img).text((pad - bbox[0], pad - bbox[1]), ch, fill=255, font=font)
    arr = np.array(img)
    ys, xs = np.where(arr > 0)
    if len(ys) == 0:
        return arr
    return arr[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

def rotate_binary(img, angle_deg):
    h, w = img.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle_deg, 1.0)
    rot = cv2.warpAffine(img, M, (w, h), borderValue=0)
    ys, xs = np.where(rot > 0)
    if len(ys) == 0:
        return rot
    return rot[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

def build_templates():
    fonts = find_available_fonts()
    print(f'Fonts en ús ({len(fonts)}):')
    for f in fonts:
        print(f'  - {f}')
    plantilles = {}  # {char: [rotacions × fonts]}
    for ch in ALFABET:
        all_variants = []
        for fp in fonts:
            try:
                base = render_char(ch, fp)
                for a in ANGLES_DEG:
                    all_variants.append(rotate_binary(base, a))
            except Exception:
                continue
        plantilles[ch] = all_variants
    return plantilles

PLANTILLES = build_templates()
total = sum(len(v) for v in PLANTILLES.values())
print(f'\n{len(PLANTILLES)} caràcters × {total // len(PLANTILLES)} variants = {total} plantilles')

### Visualització d'algunes plantilles

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(16, 5))
chars_sample = list(ALFABET)[:30]
for ax, ch in zip(axes.flat, chars_sample):
    ax.imshow(PLANTILLES[ch][3])  # rotació 0°
    ax.set_title(ch, fontsize=10)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 9. Reconeixement per template matching

Per cada caràcter segmentat:
1. Provem cada plantilla en cada rotació.
2. Redimensionem el candidat a la mida de la plantilla.
3. Calculem la **correlació normalitzada** (`TM_CCOEFF_NORMED` d'OpenCV, equivalent a `funcion_CorrelacionEntreMatrices` de UHU sense desplaçament).
4. Ens quedem amb el (caràcter, rotació) de correlació màxima.

També retornem la confiança de cada predicció.

In [ ]:
def recognize_char(roi, plantilles=PLANTILLES):
    if roi.size == 0:
        return ' ', 0.0
    best_score = -np.inf
    best_char  = '?'
    for ch, rotacions in plantilles.items():
        for tpl in rotacions:
            if tpl.size == 0:
                continue
            # Redimensionem el candidat a la mida de la plantilla
            cand = cv2.resize(roi, (tpl.shape[1], tpl.shape[0]),
                              interpolation=cv2.INTER_NEAREST)
            # Correlació normalitzada (CCOEFF_NORMED, equivalent a Pearson 2D)
            res = cv2.matchTemplate(cand, tpl, cv2.TM_CCOEFF_NORMED)
            score = float(res[0, 0])
            if score > best_score:
                best_score = score
                best_char  = ch
    return best_char, best_score

def recognize_plate(rois):
    chars, scores = [], []
    for roi in rois:
        c, s = recognize_char(roi)
        chars.append(c)
        scores.append(s)
    return ''.join(chars), scores

## 10. Pipeline complet amb rebuig multi-país

`process_crop` itera sobre `NUM_OBJETOS_TRY = [7, 6, 8, 5]` i prova cada valor. Per cada N que dóna exactament N caràcters segmentats:

1. Reconeix els caràcters per template matching.
2. Valida amb `validate_plate_text()`:
   - Llargada entre 5 i 8 caràcters.
   - Mix de lletres **i** dígits (descarta "MERCEDES" o "12345").
   - Cap caràcter pot ocupar > 60 % (descarta "EEEEEE").
   - Coincideix amb almenys un patró: ES, FR/IT, RS, PL, CZ, DE, o el fallback genèric EU.
3. Confiança mitjana ≥ `MIN_CONFIDENCE`.

De tots els candidats acceptables, es queda el millor (prioritat: patró específic > genèric > confiança).

Si cap N produeix un candidat vàlid → **rebutjat** amb el motiu més informatiu possible.

In [ ]:
def validate_plate_text(text):
    """Valida una cadena reconeguda contra els patrons de país i regles estructurals.

    Retorna (is_valid: bool, label: str) on label és el país que ha coincidit
    o el motiu de rebuig (length, no_mix, repetitive, no_pattern).
    """
    if not text or not (5 <= len(text) <= 8):
        return False, 'length'
    n_letters = sum(c.isalpha() for c in text)
    n_digits  = sum(c.isdigit() for c in text)
    if n_letters == 0 or n_digits == 0:
        return False, 'no_mix'
    # Cap caràcter pot ocupar més del 60 % (descarta soroll repetitiu tipus 'EEEEEEE')
    if max(Counter(text).values()) > len(text) * 0.6:
        return False, 'repetitive'
    for country, pat in PLATE_PATTERNS.items():
        if pat.match(text):
            return True, country
    return False, 'no_pattern'


def process_crop(img_bgr):
    """Pipeline complet amb rebuig multi-país.

    Prova els valors de NUM_OBJETOS_TRY en ordre i es queda amb el millor
    candidat acceptat (prioritat: patró específic > patró genèric > confiança).
    """
    best = None

    for n in NUM_OBJETOS_TRY:
        seg = segment_plate(img_bgr, n)
        if seg['n_chars'] != n:
            continue

        plate, scores = recognize_plate(seg['rois'])
        ok, label = validate_plate_text(plate)
        conf = float(np.mean(scores)) if scores else 0.0

        if not ok or conf < MIN_CONFIDENCE:
            continue

        # Score de selecció: (especificitat_patró, confiança)
        specificity = 0 if label == 'generic_eu' else 1
        rank = (specificity, conf)
        cand = {
            'accepted':   True,
            'reason':     label,
            'plate':      plate,
            'scores':     scores,
            'confidence': conf,
            'n_objetos':  n,
            'seg':        seg,
            '_rank':      rank,
        }
        if best is None or rank > best['_rank']:
            best = cand

    if best is not None:
        best.pop('_rank')
        return best

    # Cap candidat acceptable — re-fem segmentació amb N=7 per visualitzar el rebuig
    seg = segment_plate(img_bgr, 7)
    if seg['n_chars'] == 0:
        reason = 'no_chars'
    elif seg['n_chars'] != 7:
        reason = f'n_chars={seg["n_chars"]}'
    else:
        plate, scores = recognize_plate(seg['rois'])
        ok, label = validate_plate_text(plate)
        conf = float(np.mean(scores)) if scores else 0.0
        if conf < MIN_CONFIDENCE:
            reason = f'low_conf={conf:.2f}'
        else:
            reason = f'{label}: "{plate}"'
        return {
            'accepted': False, 'reason': reason,
            'plate': plate, 'scores': scores, 'confidence': conf, 'seg': seg,
        }
    return {
        'accepted': False, 'reason': reason,
        'plate': None, 'scores': None, 'confidence': 0.0, 'seg': seg,
    }

## 11. Prova sobre una mostra

Visualitzem el resultat sobre alguns retalls: mostrem original + caràcters segmentats + text reconegut + veredicte (ACCEPTAT / REBUTJAT).

In [ ]:
def show_result(img_bgr, result, name=''):
    seg = result['seg']
    n_chars = max(1, seg['n_chars'])
    fig, axes = plt.subplots(1, n_chars + 1, figsize=(2 + 1.2 * n_chars, 3))
    if n_chars + 1 == 1:
        axes = [axes]
    axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0].axis('off')

    if result['accepted']:
        title = f'{name}\n✓ {result["plate"]} (conf={result["confidence"]:.2f})'
        color = 'green'
    else:
        title = f'{name}\n✗ {result["reason"]}'
        color = 'red'
    axes[0].set_title(title, color=color, fontsize=9)

    if seg['n_chars'] > 0:
        for i, roi in enumerate(seg['rois']):
            axes[i + 1].imshow(roi); axes[i + 1].axis('off')
            if result['plate'] and i < len(result['plate']):
                axes[i + 1].set_title(result['plate'][i], fontsize=11)
    plt.tight_layout(); plt.show()

for p in random.sample(crops, 8):
    img = cv2.imread(str(p))
    res = process_crop(img)
    show_result(img, res, name=p.name)

## 12. Execució sobre tots els retalls + estadístiques

Apliquem el pipeline a tots els crops de `data/processed/`. Esperem que la majoria siguin rebutjats (els fals positius del detector) i que ens quedin només els que tenen una matrícula reconeixible amb format ES vàlid.

In [ ]:
results = []
for p in crops:
    img = cv2.imread(str(p))
    if img is None:
        continue
    r = process_crop(img)
    r['path'] = p
    results.append(r)

n_total = len(results)
n_acc   = sum(1 for r in results if r['accepted'])
n_rej   = n_total - n_acc

acc_by_country = Counter()
rej_reasons    = Counter()

for r in results:
    if r['accepted']:
        acc_by_country[r['reason']] += 1
    else:
        reason = r['reason']
        # Agrupem motius equivalents per llegibilitat
        if reason.startswith('n_chars='):
            rej_reasons['n_chars_incorrecte'] += 1
        elif reason.startswith('low_conf'):
            rej_reasons['baixa_confiança'] += 1
        elif reason in ('length', 'no_mix', 'repetitive', 'no_pattern'):
            rej_reasons[f'patró_{reason}'] += 1
        elif reason == 'no_chars':
            rej_reasons['cap_caràcter'] += 1
        else:
            rej_reasons[reason] += 1

print(f'Total retalls:     {n_total}')
print(f'Acceptats:         {n_acc} ({100*n_acc/n_total:.1f}%)')
print(f'Rebutjats:         {n_rej} ({100*n_rej/n_total:.1f}%)')
print('\nAcceptats per patró:')
for c, n in acc_by_country.most_common():
    print(f'  {c:20s} {n}')
print('\nMotius de rebuig:')
for c, n in rej_reasons.most_common():
    print(f'  {c:25s} {n}')

## 13. Matrícules acceptades

Llistem les matrícules acceptades agrupades per imatge original (el nom abans de `_box`). Si una imatge té múltiples retalls acceptats, ens quedem amb el de **major confiança**.

In [ ]:
from collections import defaultdict

per_image = defaultdict(list)
for r in results:
    if r['accepted']:
        stem = r['path'].stem.rsplit('_box', 1)[0]
        per_image[stem].append(r)

final = {}
for stem, lst in per_image.items():
    best = max(lst, key=lambda r: r['confidence'])
    final[stem] = (best['plate'], best['confidence'])

print(f'Imatges originals amb almenys una matrícula vàlida: {len(final)}\n')
for stem in sorted(final):
    plate, conf = final[stem]
    print(f'  {stem:20s} → {plate}  (conf={conf:.2f})')

## 14. Visualització de les acceptades

Mostrem totes les matrícules acceptades amb el text reconegut a sota.

In [ ]:
accepted = [r for r in results if r['accepted']]
n = len(accepted)
if n == 0:
    print('Cap matrícula acceptada.')
else:
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, 2.5 * rows))
    axes = np.atleast_2d(axes).flatten()
    for ax, r in zip(axes, accepted):
        img = cv2.imread(str(r['path']))
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f'{r["plate"]} ({r["confidence"]:.2f})', fontsize=9, color='green')
        ax.axis('off')
    for ax in axes[n:]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

## 15. Següents passos

- **Afegir FE-Schrift al projecte** (`fonts/FE-Schrift.ttf`) per millorar significativament el matching en matrícules EU centrals (DE, GR, IT, ES recents, PL...).
- **Calibrar `MIN_CONFIDENCE`**: si el rebuig per baixa confiança és alt, baixa el llindar; si veus falsos positius (text de carrosseria reconegut com a matrícula), puja'l.
- **Afegir més patrons per país** si en veus alguns sistemàticament rebutjats per `no_pattern` (mira la cel·la 27 per estadístiques de motius).
- **Matriu de confusió** sobre les acceptades per detectar parelles problemàtiques (0↔O, 1↔I, 5↔S, 8↔B, 2↔Z) i ajustar plantilles o post-processar.